# PKG Attrition — Source Profiling & Metric Build · v4

**Full-window run.** v3 profiled three months and answered the open questions.
This version widens to the whole panel, fixes what the v3 run exposed, and
promotes the two findings worth building on.

### What v3 established (now treated as known, not re-derived)

| Fact | Value | Consequence |
|---|---|---|
| Counterparty key provenance | 96.3% of dollars **account-derived** (`RTN-account`) | stable nodes; Group B needs no caveat |
| Identifiability | 97.5% of off-us dollars carry key + name + bank | Group A is live |
| Coverage of deposit book | 80.8% any txn · 65.0% off-us outbound | replaces the ~12% figure |
| Topology | OUT 52.2% legs / 40.3% $ · IN 41.8% / 40.3% · INTERNAL 6.0% / 19.4% · ORPHAN 0 | the on-us graph saw 19.4% of dollars |
| Rail | WIRE 0.5% legs / 58.2% $ · ACH 86.3% / 39.3% | every dollar rule must be rail-relative |
| `legs_per_txn` | 0.999 | no double-booking |

### What v3 broke, and what changed here

| v3 problem | Fix in v4 |
|---|---|
| `txn_silence` returned rate **1.000** — 30 deposit months against 3 txn months | silence computed only inside the transaction-covered window (E6) |
| Deposit table has 66,947 rows for 66,733 customers | deterministic de-dup with a conflict count (D1) |
| B2 + B4b were 560 s of 1,126 s and are one-time taxonomy work | `RUN_TAXONOMY` flag, default **off** |
| `corr_delta_netflow = 0.04` reported bare | re-tested on gross flows and log balance before any conclusion (D4) |
| Rule precision computed by hand | 2×2 confusion matrix is a first-class output (E5) |
| Survivorship suspected, not tested | dedicated diagnostic (D2b) |

### What is new

- **E1 · `off_pnc_ach_orig_share`** — the share of a customer's *outbound* ACH
  originated through a non-PNC bank. $144.8B of $707.2B outbound ACH in three
  months (20.5%). A direct treasury-services wallet-share ratio, no name
  matching, no fuzzy resolution. Stronger than the brief's priority signal.
- **E2 · same-name as a delta**, with the tax/treasury false-positive
  mechanism excluded.
- **E3 · counterparty churn** — Group B feasibility, cheap, from `edges_monthly`.

In [ ]:
# ============================================================================
# CONFIG
# ============================================================================
import os, re, json, time
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window as W

CONFIG = dict(
    TXN_TABLE  = "<db>.<staging_transactions>",
    DEP_TABLE  = "<db>.<deposit_wide>",
    CUST_TABLE = "<db>.neo4j_customer",

    WORK_DIR   = "hdfs:///user/<you>/pkg_attrition/work",
    OUT_DIR    = "../eda/attrition",

    # Full window. The deposit panel runs 2024-01 .. 2026-06; the transaction
    # window is set to match so every deposit month has a flow counterpart.
    MONTH_MIN  = "2024-01",
    MONTH_MAX  = "2026-06",

    PARTY_TYPES     = ["O"],
    REQUIRE_DEPOSIT = True,

    # --- stage control -------------------------------------------------------
    # Taxonomy profiling (rail / category / identifiability / hub seed) was
    # settled in v3 and will not change. 560 s of the 1,126 s v3 run. Off by
    # default; turn on only to re-validate after a schema change.
    RUN_TAXONOMY = False,
    REBUILD      = False,

    # --- domain --------------------------------------------------------------
    BALANCE_FLOOR     = 25_000,
    CURRENT_RULE_DROP = 0.30,
    SILENCE_MONTHS    = 3,
    HLL_RSD           = 0.01,
    SAMPLE_FRAC       = 0.001,

    # Rails with no counterparty key at all (v3 §B3). Excluded from counterparty
    # metrics explicitly rather than dropping out silently — they still count
    # toward flow.
    NO_KEY_RAILS = ["PCARD", "DEBIT_CARD_SIGNATURE"],
)

# Confirmed taxonomy from the v3 run. Used as an assertion, not a discovery:
# a value outside these sets means the source changed and the run should stop.
KNOWN = dict(
    RAILS      = {"WIRE", "ACH", "CHECK", "RTP_PRT", "PCARD", "DEBIT_CARD_SIGNATURE", "RTP_P2P"},
    CPTY_TYPES = {"CPTY", "NON_PNC_ACH_ORIGINATOR", "MERCHANT", "P2P_CPTY"},
    TOPOLOGY   = {"INTERNAL_C2C", "INBOUND", "OUTBOUND", "ORPHAN"},
)

T = dict(
    pays_id="mdm_id_pays", pays_name="customer_name_pays", pays_acct="pnc_dep_acct_pays",
    recv_id="mdm_id_receives", recv_name="customer_name_receives", recv_acct="pnc_dep_acct_receives",
    cpty_key="unq_cpty_acct_id", cpty_name="cpty_name", cpty_type="cpty_type",
    cpty_fi="cpty_fin_entity_name",
    txn_id="trans_id", amount="trans_amt", currency="trans_currency", date="trans_dt",
    rail="payment_rail", category="category", cat_prefix="category_prefix", src_syst="src_syst",
    merch_id="merchant_id",
)
DEP  = dict(cust="cust_pwr_id", rltn="rltn_pwr_id", latest="latest_month",
            prior_avg="prior_avg", recent_avg="recent_avg", pct_change="pct_change",
            peak="peak_balance", latest_bal="latest_month_bal",
            n_acct="latest_num_accounts", n_closed="latest_num_closed_accounts")
CUST = dict(mdm_id="mdm_id", pwr_id="cust_pwr_id", party="party_type",
            naics="naics_cd", name="customer_name")

# Tax and government payment names that carry the PAYER's own name in the
# counterparty field — the same-name false-positive mechanism found in v3.
TAX_FP_PATTERN = (r"(TREASURY|IRS|EFTPS|TAXPAYOR|TAXPAYER|DEPT\s*REVENUE|"
                  r"FRANCHISE\s*TAX|DEPT\s*TAXATION|WEBFILE|BWC)")

In [ ]:
# ============================================================================
# SPARK & HELPERS
# ============================================================================
spark = SparkSession.builder.getOrCreate()
for k, v in {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.skewJoin.enabled": "true",
    "spark.sql.adaptive.localShuffleReader.enabled": "true",
    "spark.sql.shuffle.partitions": "2000",
    "spark.sql.parquet.filterPushdown": "true",
    "spark.sql.execution.arrow.pyspark.enabled": "true",
    "spark.sql.execution.arrow.pyspark.fallback.enabled": "true",
    "spark.sql.autoBroadcastJoinThreshold": str(64 * 1024 * 1024),
    "spark.sql.sources.partitionOverwriteMode": "dynamic",
}.items():
    spark.conf.set(k, v)

OUT, WORK = CONFIG["OUT_DIR"], CONFIG["WORK_DIR"]
os.makedirs(OUT, exist_ok=True)
R, TIMINGS = {}, []


def month_range(lo, hi):
    out, y, m = [], int(lo[:4]), int(lo[5:7])
    while f"{y:04d}-{m:02d}" <= hi:
        out.append(f"{y:04d}-{m:02d}")
        m += 1
        if m == 13:
            y, m = y + 1, 1
    return out

MONTHS = month_range(CONFIG["MONTH_MIN"], CONFIG["MONTH_MAX"])
print(f"{len(MONTHS)} months: {MONTHS[0]} .. {MONTHS[-1]}")


class timed:
    def __init__(self, label): self.label = label
    def __enter__(self): self.t = time.time(); print(f"\n>>> {self.label}"); return self
    def __exit__(self, *a):
        dt = time.time() - self.t
        TIMINGS.append({"stage": self.label, "seconds": round(dt, 1)})
        print(f"<<< {self.label}: {dt/60:.1f} min")


def save(pdf, name, note=""):
    if len(pdf) > 5000:
        raise ValueError(f"{name}: {len(pdf)} rows is not a summary")
    pdf.to_csv(os.path.join(OUT, f"{name}.csv"), index=False)
    print(f"\n--- {name}{' · ' + note if note else ''}")
    with pd.option_context("display.max_rows", 80, "display.width", 220,
                           "display.float_format", lambda v: f"{v:,.4f}"):
        print(pdf.to_string(index=False))
    return pdf


def path(n): return f"{WORK}/{n}"

def exists(n):
    try:
        spark.read.parquet(path(n)).limit(1).count(); return True
    except Exception:
        return False

def present(c):
    col = F.col(c) if isinstance(c, str) else c
    return col.isNotNull() & (F.trim(col.cast("string")) != "")

def ndv(c, rsd=None): return F.approx_count_distinct(c, rsd or CONFIG["HLL_RSD"])

def norm_name(col):
    c = F.upper(F.trim(col))
    c = F.regexp_replace(c, r"[^A-Z0-9 ]", " ")
    c = F.regexp_replace(c, r"\b(LLC|L L C|INC|INCORPORATED|CORP|CORPORATION|CO|LP|LLP|"
                            r"PLLC|PC|LTD|LIMITED|DBA|THE)\b", " ")
    return F.trim(F.regexp_replace(c, r"\s+", " "))

def prev_month(col):
    """'YYYY-MM' minus one month, as a string."""
    return F.date_format(F.add_months(F.to_date(F.concat(col, F.lit("-01"))), -1), "yyyy-MM")

def next_month(col):
    """'YYYY-MM' plus one month, as a string. Used to align the lag self-join."""
    return F.date_format(F.add_months(F.to_date(F.concat(col, F.lit("-01"))), 1), "yyyy-MM")

def dshare(df, cols, name, note="", amt="amount"):
    g = (df.groupBy(*cols).agg(F.count(F.lit(1)).alias("n_legs"),
                               F.sum(amt).alias("dollars")).toPandas())
    g["share_legs"] = g["n_legs"] / g["n_legs"].sum()
    g["share_dollars"] = g["dollars"] / g["dollars"].sum()
    return save(g.sort_values("dollars", ascending=False), name, note)

print("ready ·", WORK)

---
# PHASE A — Build once

## A2 · Corporate key set

Organisations with a `cust_pwr_id` present in the deposit book. The 1:1
assertion is checked rather than assumed: a fanout means one deposit-table
"customer" is several graph nodes and every per-customer aggregate is wrong by
an unknown factor.

In [ ]:
with timed("A1 partition check"):
    try:
        parts = spark.sql(f"SHOW PARTITIONS {CONFIG['TXN_TABLE']}").toPandas()
        print(f"PARTITIONED · {len(parts)} partitions")
        R["partitioned"] = True
    except Exception:
        print("*** NOT PARTITIONED — every staging read is a full scan. ***")
        R["partitioned"] = False

In [ ]:
with timed("A2 corporate key set"):
    if CONFIG["REBUILD"] or not exists("corp_keys"):
        cust = spark.table(CONFIG["CUST_TABLE"])
        corp = (cust.filter(F.col(CUST["party"]).isin(CONFIG["PARTY_TYPES"]) & present(CUST["pwr_id"]))
                    .select(F.col(CUST["mdm_id"]).alias("mdm_id"),
                            F.col(CUST["pwr_id"]).alias("cust_pwr_id"),
                            F.col(CUST["naics"]).alias("naics_cd"),
                            norm_name(F.col(CUST["name"])).alias("ego_name_norm"))
                    .dropDuplicates(["mdm_id"]))
        if CONFIG["REQUIRE_DEPOSIT"]:
            dep_ids0 = (spark.table(CONFIG["DEP_TABLE"])
                             .select(F.col(DEP["cust"]).alias("cust_pwr_id")).distinct())
            corp = corp.join(F.broadcast(dep_ids0), "cust_pwr_id")
        corp.write.mode("overwrite").parquet(path("corp_keys"))

    corp = spark.read.parquet(path("corp_keys"))
    R["n_corporate_egos"] = corp.count()
    print(f"corporate egos: {R['n_corporate_egos']:,}")

    save(corp.groupBy("cust_pwr_id").agg(F.count(F.lit(1)).alias("n_mdm"))
             .groupBy("n_mdm").count().orderBy("n_mdm").limit(10).toPandas(),
         "A2_mdm_per_pwr_id", "*** n_mdm > 1 breaks the 1:1 assumption ***")

## A3 · The one pass — ego legs

The only read of raw staging. Projects, prunes to the window, filters to
corporate egos, explodes to legs, writes parquet partitioned by month.

**New in v4:** `key_provenance` and the ACH-origination class are derived here
rather than downstream, so every consumer of `legs` sees the same definition.
The provenance regex was confirmed against 40 sampled keys in v3
(`RTN-account`, e.g. `021000021-00000000000454247219`).

In [ ]:
with timed("A3 build ego legs — THE ONE PASS"):
    if CONFIG["REBUILD"] or not exists("legs"):
        raw = (spark.table(CONFIG["TXN_TABLE"])
               .withColumn("month", F.substring(F.col(T["date"]), 1, 7))
               .filter(F.col("month").isin(MONTHS))
               .select("month",
                       F.col(T["txn_id"]).alias("txn_id"),
                       F.col(T["date"]).alias("trans_dt"),
                       F.col(T["amount"]).cast("double").alias("amount"),
                       F.col(T["currency"]).alias("currency"),
                       F.col(T["rail"]).alias("rail"),
                       F.col(T["category"]).alias("category"),
                       F.col(T["cat_prefix"]).alias("cat_prefix"),
                       F.col(T["pays_id"]).alias("pays_id"), F.col(T["pays_name"]).alias("pays_name"),
                       F.col(T["pays_acct"]).alias("pays_acct"),
                       F.col(T["recv_id"]).alias("recv_id"), F.col(T["recv_name"]).alias("recv_name"),
                       F.col(T["recv_acct"]).alias("recv_acct"),
                       F.col(T["cpty_key"]).alias("cpty_raw"), F.col(T["cpty_name"]).alias("cpty_name"),
                       F.col(T["cpty_type"]).alias("cpty_type"), F.col(T["cpty_fi"]).alias("cpty_fi"),
                       F.col(T["merch_id"]).alias("merch_id")))

        raw = (raw.withColumn("_p", present("pays_id")).withColumn("_r", present("recv_id"))
                  .withColumn("topology",
                      F.when(F.col("_p") & F.col("_r"), "INTERNAL_C2C")
                       .when(~F.col("_p") & F.col("_r"), "INBOUND")
                       .when(F.col("_p") & ~F.col("_r"), "OUTBOUND").otherwise("ORPHAN"))
                  .withColumn("is_self_loop",
                      F.col("_p") & F.col("_r") & (F.col("pays_id") == F.col("recv_id"))))

        leg_out = F.when(F.col("_p"), F.struct(
            F.lit("OUT").alias("ego_dir"),
            F.col("pays_id").alias("ego_mdm"), F.col("pays_name").alias("ego_name"),
            F.col("pays_acct").alias("ego_acct"),
            F.coalesce(F.concat(F.lit("PNC:"), F.col("recv_id")), F.col("cpty_raw")).alias("cpty_key"),
            F.when(F.col("_r"), F.lit("on_us")).otherwise(F.lit("off_us")).alias("cpty_key_src"),
            F.coalesce(F.col("recv_name"), F.col("cpty_name")).alias("cpty_name_eff")))
        leg_in = F.when(F.col("_r"), F.struct(
            F.lit("IN").alias("ego_dir"),
            F.col("recv_id").alias("ego_mdm"), F.col("recv_name").alias("ego_name"),
            F.col("recv_acct").alias("ego_acct"),
            F.coalesce(F.concat(F.lit("PNC:"), F.col("pays_id")), F.col("cpty_raw")).alias("cpty_key"),
            F.when(F.col("_p"), F.lit("on_us")).otherwise(F.lit("off_us")).alias("cpty_key_src"),
            F.coalesce(F.col("pays_name"), F.col("cpty_name")).alias("cpty_name_eff")))

        legs = (raw.withColumn("leg", F.explode(F.array(leg_out, leg_in)))
            .filter(F.col("leg").isNotNull())
            .select("month", "txn_id", "trans_dt", "amount", "currency", "rail", "category",
                    "cat_prefix", "topology", "is_self_loop",
                    "cpty_raw", "cpty_name", "cpty_type", "cpty_fi", "merch_id",
                    F.col("leg.ego_dir").alias("ego_dir"),
                    F.col("leg.ego_mdm").alias("mdm_id"),
                    F.col("leg.ego_name").alias("ego_name"),
                    F.col("leg.ego_acct").alias("ego_acct"),
                    F.col("leg.cpty_key").alias("cpty_key"),
                    F.col("leg.cpty_key_src").alias("cpty_key_src"),
                    F.col("leg.cpty_name_eff").alias("cpty_name_eff"))
            # Derived once, here, so every consumer shares the definition.
            .withColumn("key_provenance",
                F.when(~present("cpty_key"), "none")
                 .when(F.col("cpty_key").startswith("PNC:"), "on_us")
                 .when(F.col("cpty_key").rlike(r"^[0-9][0-9\-_|]*[0-9]$"), "account_derived")
                 .otherwise("name_derived"))
            # The treasury-services axis: outbound ACH the customer originated,
            # split by whether PNC or another bank carried the origination.
            .withColumn("ach_orig_class",
                F.when((F.col("rail") == "ACH") & (F.col("ego_dir") == "OUT")
                       & (F.col("cpty_type") == "NON_PNC_ACH_ORIGINATOR"), "out_ach_nonpnc")
                 .when((F.col("rail") == "ACH") & (F.col("ego_dir") == "OUT"), "out_ach_pnc")
                 .when((F.col("rail") == "ACH") & (F.col("ego_dir") == "IN")
                       & (F.col("cpty_type") == "NON_PNC_ACH_ORIGINATOR"), "in_ach_nonpnc")
                 .when((F.col("rail") == "ACH") & (F.col("ego_dir") == "IN"), "in_ach_pnc")
                 .otherwise("not_ach"))
            .join(F.broadcast(corp.select("mdm_id", "cust_pwr_id", "naics_cd", "ego_name_norm")),
                  "mdm_id"))

        legs.write.mode("overwrite").partitionBy("month").parquet(path("legs"))

    legs = spark.read.parquet(path("legs"))
    print("legs ·", path("legs"))

In [ ]:
with timed("A4 sanity + taxonomy assertion"):
    save(legs.groupBy("month").agg(
            F.count(F.lit(1)).alias("n_legs"),
            ndv("cust_pwr_id").alias("n_egos"),
            F.sum("amount").alias("dollars")).orderBy("month").toPandas(),
         "A4_monthly_volume", "a ramp at either end is an ingestion artefact")

    # Assertion, not discovery. v3 settled the taxonomy; an unseen value means
    # the source changed and everything downstream is suspect.
    for col, known in [("rail", KNOWN["RAILS"]), ("cpty_type", KNOWN["CPTY_TYPES"]),
                       ("topology", KNOWN["TOPOLOGY"])]:
        seen = {r[0] for r in legs.select(col).distinct().collect() if r[0] is not None}
        new = seen - known
        print(f"{col}: {len(seen)} values" + (f"  *** UNEXPECTED: {new} ***" if new else "  ok"))
        if new:
            R[f"unexpected_{col}"] = str(new)

    one = MONTHS[len(MONTHS) // 2]
    s = legs.filter(F.col("month") == one).agg(
            F.count(F.lit(1)).alias("n_legs"), ndv("txn_id").alias("n_txn")).toPandas()
    R["legs_per_txn"] = float(s["n_legs"].iloc[0] / s["n_txn"].iloc[0])
    print(f"legs_per_txn ({one}): {R['legs_per_txn']:.4f}")

---
# PHASE B — Taxonomy profiling *(skippable)*

Settled in the v3 run and unchanged by widening the window. `RUN_TAXONOMY`
defaults to **off** — these two stages were 560 s of a 1,126 s three-month run
and scale linearly. Turn on only to re-validate after a schema change.

In [ ]:
if CONFIG["RUN_TAXONOMY"]:
    with timed("B taxonomy + identifiability + hubs"):
        dshare(legs, ["topology"], "B_topology_mix")
        dshare(legs, ["ego_dir", "cpty_key_src"], "B_direction_x_scope")
        dshare(legs, ["rail"], "B_rail")
        dshare(legs, ["category"], "B_category")
        dshare(legs, ["cpty_type"], "B_cpty_type")

        ext = legs.filter("cpty_key_src='off_us'")
        dshare(ext, ["key_provenance"], "B_key_provenance")
        dshare(ext, ["rail", "key_provenance"], "B_provenance_by_rail")
        save(ext.filter(present("cpty_key")).select("cpty_key", "key_provenance", "rail")
                .sample(CONFIG["SAMPLE_FRAC"]).limit(40).toPandas(),
             "B_key_format_sample", "confirm the provenance regex")

        save(ext.filter(present("cpty_fi")).groupBy("cpty_fi")
                .agg(ndv("cust_pwr_id").alias("n_customers"), F.sum("amount").alias("dollars"))
                .orderBy(F.desc("dollars")).limit(100).toPandas(),
             "B_top_financial_entities", "the competitive map")
        save(ext.filter(present("cpty_name_eff"))
                .groupBy(norm_name(F.col("cpty_name_eff")).alias("norm_name"))
                .agg(ndv("cust_pwr_id").alias("n_customers"), F.sum("amount").alias("dollars"),
                     F.first("cpty_type", ignorenulls=True).alias("type_sample"))
                .orderBy(F.desc("n_customers")).limit(100).toPandas(),
             "B_top_counterparty_names", "hub taxonomy seed")
else:
    print("B skipped (RUN_TAXONOMY=False). v3 results stand — see the header table.")

---
# PHASE C — Derived tables

`edges_monthly`, `cust_monthly`, `cpty_dim`. Everything after Phase C reads
these, never `legs`, and never staging.

In [ ]:
with timed("C1 edges_monthly"):
    if CONFIG["REBUILD"] or not exists("edges_monthly"):
        (legs.filter("NOT is_self_loop")
             .groupBy("cust_pwr_id", "cpty_key", "cpty_key_src", "ego_dir", "month")
             .agg(F.sum("amount").alias("amount"),
                  F.count(F.lit(1)).alias("volume"),
                  F.min("trans_dt").alias("first_dt"), F.max("trans_dt").alias("last_dt"),
                  F.first("cpty_fi", ignorenulls=True).alias("cpty_fi"),
                  F.first("cpty_name_eff", ignorenulls=True).alias("cpty_name"),
                  F.first("cpty_type", ignorenulls=True).alias("cpty_type"),
                  F.first("key_provenance", ignorenulls=True).alias("key_provenance"))
             .write.mode("overwrite").partitionBy("month").parquet(path("edges_monthly")))
    edges = spark.read.parquet(path("edges_monthly"))

In [ ]:
with timed("C2 cust_monthly"):
    if CONFIG["REBUILD"] or not exists("cust_monthly"):
        (legs.filter("NOT is_self_loop")
             .groupBy("cust_pwr_id", "month")
             .agg(
                 F.sum(F.when(F.col("ego_dir") == "OUT", F.col("amount")).otherwise(0.0)).alias("gross_out"),
                 F.sum(F.when(F.col("ego_dir") == "IN",  F.col("amount")).otherwise(0.0)).alias("gross_in"),
                 F.sum(F.when((F.col("ego_dir") == "OUT") & (F.col("cpty_key_src") == "off_us"),
                              F.col("amount")).otherwise(0.0)).alias("off_us_out"),
                 # --- ACH origination block (new in v4) ----------------------
                 F.sum(F.when(F.col("ach_orig_class") == "out_ach_nonpnc", F.col("amount")).otherwise(0.0)).alias("out_ach_nonpnc"),
                 F.sum(F.when(F.col("ach_orig_class") == "out_ach_pnc",    F.col("amount")).otherwise(0.0)).alias("out_ach_pnc"),
                 F.sum(F.when(F.col("ach_orig_class") == "in_ach_nonpnc",  F.col("amount")).otherwise(0.0)).alias("in_ach_nonpnc"),
                 F.sum(F.when(F.col("ach_orig_class") == "in_ach_pnc",     F.col("amount")).otherwise(0.0)).alias("in_ach_pnc"),
                 F.count(F.lit(1)).alias("n_legs"),
                 ndv("cpty_key").alias("n_cpty"),
                 ndv(F.when(F.col("ego_dir") == "OUT", F.col("cpty_key"))).alias("n_cpty_out"),
                 ndv("ego_acct").alias("n_active_accts"))
             .withColumn("net_flow", F.col("gross_in") - F.col("gross_out"))
             .withColumn("off_us_out_share", F.col("off_us_out") / F.greatest(F.col("gross_out"), F.lit(1.0)))
             # THE metric: of the outbound ACH this customer originated, what
             # share went through a bank that is not PNC. Rising = losing
             # treasury services. NaN where there is no outbound ACH at all —
             # deliberately not zero, which would read as "all with PNC".
             .withColumn("out_ach_total", F.col("out_ach_nonpnc") + F.col("out_ach_pnc"))
             .withColumn("off_pnc_ach_orig_share",
                 F.when(F.col("out_ach_total") > 0,
                        F.col("out_ach_nonpnc") / F.col("out_ach_total")))
             .write.mode("overwrite").partitionBy("month").parquet(path("cust_monthly")))
    cm = spark.read.parquet(path("cust_monthly"))

In [ ]:
with timed("C3 cpty_dim"):
    if CONFIG["REBUILD"] or not exists("cpty_dim"):
        # Two-stage reduce: distinct (cpty, ego) pairs first, then count. This is
        # what keeps a hub with hundreds of millions of legs off a single task.
        pairs = legs.filter("cpty_key_src='off_us'").select("cpty_key", "cust_pwr_id").distinct()
        attrs = (legs.filter("cpty_key_src='off_us'").groupBy("cpty_key")
                 .agg(F.first("cpty_name_eff", ignorenulls=True).alias("cpty_name"),
                      F.first("cpty_fi", ignorenulls=True).alias("cpty_fi"),
                      F.first("cpty_type", ignorenulls=True).alias("cpty_type"),
                      F.first("key_provenance", ignorenulls=True).alias("key_provenance"),
                      F.sum("amount").alias("dollars"), F.count(F.lit(1)).alias("n_legs"),
                      F.min("month").alias("first_month"), F.max("month").alias("last_month")))
        (pairs.groupBy("cpty_key").agg(F.count(F.lit(1)).alias("n_customers"))
              .join(attrs, "cpty_key").write.mode("overwrite").parquet(path("cpty_dim")))
    cpd = spark.read.parquet(path("cpty_dim"))

    fan = cpd.agg(F.count(F.lit(1)).alias("n_counterparties"),
                  F.mean((F.col("n_customers") >= 2).cast("double")).alias("share_2plus"),
                  F.mean((F.col("n_customers") >= 5).cast("double")).alias("share_5plus"),
                  F.max("n_customers").alias("max_customers")).toPandas()
    save(fan, "C3_cpty_fanin",
         "*** share_5plus bounds Group C. v3 measured 0.0020 — re-read at full window. ***")
    R["cpty_share_5plus"] = float(fan["share_5plus"].iloc[0])

---
# PHASE D — Deposit panel

## D1 · Unpivot, with de-duplication

v3 found 66,947 rows against 66,733 distinct `cust_pwr_id`. Left alone, those
214 duplicates double-weight their customers in the funnel and the label
prevalence. Resolved deterministically here: conflicts are counted first, then
the row with the most accounts wins, then the highest latest balance — never
positional `first`.

In [ ]:
with timed("D1 deposit unpivot + dedup"):
    dep_w0 = spark.table(CONFIG["DEP_TABLE"])
    bal_cols = sorted([c for c in dep_w0.columns if re.match(r"bal_\d{4}_\d{2}$", c)])
    print(f"{len(bal_cols)} balance columns: {bal_cols[0]} .. {bal_cols[-1]}")

    dup = (dep_w0.groupBy(DEP["cust"]).agg(F.count(F.lit(1)).alias("n"))
                 .filter("n > 1").agg(F.count(F.lit(1)).alias("n_dup_customers"),
                                      F.sum("n").alias("n_dup_rows")).toPandas())
    save(dup, "D1_duplicate_customers", "resolved below; not dropped silently")

    wdedup = W.partitionBy(DEP["cust"]).orderBy(
        F.col(DEP["n_acct"]).desc_nulls_last(), F.col(DEP["latest_bal"]).desc_nulls_last())
    dep_w = (dep_w0.withColumn("_rk", F.row_number().over(wdedup))
                   .filter("_rk = 1").drop("_rk"))

    if CONFIG["REBUILD"] or not exists("dep_long"):
        pairs_sql = ", ".join([f"'{c[4:].replace('_','-')}', cast({c} as double)" for c in bal_cols])
        (dep_w.select(F.col(DEP["cust"]).alias("cust_pwr_id"),
                      F.expr(f"stack({len(bal_cols)}, {pairs_sql}) as (month, balance)"))
              .write.mode("overwrite").parquet(path("dep_long")))
    dep = spark.read.parquet(path("dep_long"))

    save(dep_w.agg(F.count(F.lit(1)).alias("n_customers"),
                   ndv(DEP["rltn"]).alias("n_relationships"),
                   # A single distinct latest_month means the table is ONE snapshot
                   # as-of that month — which is the survivorship mechanism.
                   F.countDistinct(DEP["latest"]).alias("n_distinct_latest_month"),
                   F.max(DEP["latest"]).alias("max_latest_month"),
                   F.mean(F.col(DEP["n_acct"]).cast("double")).alias("mean_accounts"),
                   F.mean((F.col(DEP["n_closed"]) > 0).cast("double")).alias("share_any_closed")).toPandas(),
         "D1_deposit_shape",
         "*** n_distinct_latest_month = 1 is strong evidence of a current-book snapshot ***")

## D2 · Balance-series shapes, and the survivorship diagnostic

The v3 run showed two patterns pointing the same way: non-null share climbing
monotonically 76.0% → 100% (2025-10 onward), and **every** stop falling in the
last eight months at a flat ~600/month.

The straightforward reading is that the table is the current book with balances
backfilled — customers who left before late 2025 are not in it. If so the label
is not merely circular, it is **censored**, and episode analysis before the
cutover measures survivors only.

Three tests. Any one of them positive is enough to escalate.

In [ ]:
with timed("D2 series shapes"):
    pat = (dep.withColumn("live", F.col("balance").isNotNull() & (F.col("balance") != 0))
        .groupBy("cust_pwr_id")
        .agg(F.sum(F.col("live").cast("int")).alias("n_months_live"),
             F.min(F.when(F.col("live"), F.col("month"))).alias("first_live"),
             F.max(F.when(F.col("live"), F.col("month"))).alias("last_live"),
             F.max("month").alias("panel_end"))
        .withColumn("shape",
            F.when(F.col("n_months_live") == 0, "never_live")
             .when(F.col("last_live") == F.col("panel_end"), "live_at_end")
             .otherwise("STOPPED_BEFORE_END")))
    pat.write.mode("overwrite").parquet(path("dep_shape"))
    pat = spark.read.parquet(path("dep_shape"))

    save(pat.groupBy("shape").count().toPandas(), "D2_series_shapes")
    R["n_stopped_before_end"] = int(pat.filter("shape='STOPPED_BEFORE_END'").count())

In [ ]:
with timed("D2b SURVIVORSHIP DIAGNOSTIC"):
    # Test 1 — when do series start, and when do they stop? An unbiased panel
    # shows stops spread across the whole window. A current-book snapshot shows
    # them bunched at the recent end, because earlier leavers were removed.
    starts = pat.groupBy("first_live").agg(F.count(F.lit(1)).alias("n_start")) \
                .withColumnRenamed("first_live", "month")
    stops  = pat.filter("shape='STOPPED_BEFORE_END'") \
                .groupBy("last_live").agg(F.count(F.lit(1)).alias("n_stop")) \
                .withColumnRenamed("last_live", "month")
    save(starts.join(stops, "month", "outer").fillna(0).orderBy("month").toPandas(),
         "D2b_starts_and_stops",
         "*** stops confined to the recent tail => the panel is survivor-only ***")

    # Test 2 — concentration. What share of all stops fall in the last 8 months?
    cut = MONTHS[-8]
    conc = pat.filter("shape='STOPPED_BEFORE_END'").agg(
        F.count(F.lit(1)).alias("n_stops"),
        F.mean((F.col("last_live") >= F.lit(cut)).cast("double")).alias("share_in_last_8mo")).toPandas()
    save(conc, "D2b_stop_concentration",
         f"share_in_last_8mo near 1.0 (cut={cut}) means departures before the cutover are missing")
    R["stop_share_last_8mo"] = float(conc["share_in_last_8mo"].iloc[0])

    # Test 3 — non-null coverage by month. A monotone ramp to exactly 1.0 that
    # then never falls is the signature of a backfilled current-book extract:
    # real books lose customers, so coverage should wobble, not saturate.
    save(dep.groupBy("month").agg(
            F.mean(F.col("balance").isNotNull().cast("double")).alias("share_non_null"),
            F.mean((F.col("balance") > 0).cast("double")).alias("share_positive"),
            F.expr("percentile_approx(balance, 0.5)").alias("p50_balance"),
            F.count(F.lit(1)).alias("n")).orderBy("month").toPandas(),
         "D2b_month_coverage",
         "*** monotone ramp to 1.0 that never falls = backfilled current book ***")

    print("""
    ESCALATE IF:  stop_share_last_8mo > 0.9  OR  n_distinct_latest_month == 1
    Both say the panel cannot support historical episodes before the cutover.
    """)

## D3 · Coverage · D4 · Does the transaction table explain balance movement?

v3 reported `corr(bal_delta, net_flow) = 0.04` and that number should not be
briefed as it stands. Two reasons it is probably an artefact: the balance is a
monthly *average*, so its delta is a smoothed function of within-month flow;
and for a business gross-in ≈ gross-out, making net flow a small difference of
two very large numbers where noise dominates.

v4 tests four specifications instead of one.

In [ ]:
with timed("D3 coverage"):
    dep_ids = dep_w.select(F.col(DEP["cust"]).alias("cust_pwr_id")).distinct()
    seen    = cm.select("cust_pwr_id").distinct().withColumn("_any", F.lit(1))
    seen_o  = (edges.filter("ego_dir='OUT' AND cpty_key_src='off_us'")
                    .select("cust_pwr_id").distinct().withColumn("_out", F.lit(1)))
    cov = (dep_ids.join(seen, "cust_pwr_id", "left").join(seen_o, "cust_pwr_id", "left")
        .agg(F.count(F.lit(1)).alias("n_deposit_customers"),
             F.sum(F.coalesce("_any", F.lit(0))).alias("n_any_txn"),
             F.sum(F.coalesce("_out", F.lit(0))).alias("n_off_us_outbound")).toPandas())
    cov["coverage_any"] = cov["n_any_txn"] / cov["n_deposit_customers"]
    cov["coverage_outbound"] = cov["n_off_us_outbound"] / cov["n_deposit_customers"]
    save(cov, "D3_coverage", "*** replaces the ~12% figure, now at full window ***")
    R["deposit_txn_coverage"] = float(cov["coverage_any"].iloc[0])
    R["deposit_outbound_coverage"] = float(cov["coverage_outbound"].iloc[0])

In [ ]:
with timed("D4 flow vs balance — four specifications"):
    wd = W.partitionBy("cust_pwr_id").orderBy("month")
    panel = (dep.withColumn("bal_prev", F.lag("balance").over(wd))
        .withColumn("bal_delta", F.col("balance") - F.col("bal_prev"))
        .filter(F.col("bal_prev").isNotNull() & F.col("balance").isNotNull())
        .join(cm, ["cust_pwr_id", "month"], "left")
        .fillna({"net_flow": 0.0, "gross_out": 0.0, "gross_in": 0.0, "n_legs": 0})
        .withColumn("log_bal", F.log1p(F.greatest(F.col("balance"), F.lit(0.0))))
        .withColumn("log_out", F.log1p(F.greatest(F.col("gross_out"), F.lit(0.0))))
        .withColumn("log_in",  F.log1p(F.greatest(F.col("gross_in"),  F.lit(0.0)))))

    save(panel.agg(
            F.count(F.lit(1)).alias("n_customer_months"),
            F.corr("bal_delta", "net_flow").alias("c1_delta_vs_netflow"),
            F.corr("balance", "gross_out").alias("c2_level_vs_grossout"),
            F.corr("log_bal", "log_out").alias("c3_logbal_vs_loggrossout"),
            F.corr("log_bal", "log_in").alias("c4_logbal_vs_loggrossin")).toPandas(),
         "D4_flow_vs_balance",
         "c3/c4 are the honest test; c1 is dominated by the gross-in≈gross-out cancellation")

    save(panel.withColumn("bucket",
            F.when((F.col("n_legs") == 0) & (F.abs(F.col("bal_delta")) > 1000), "NO_TXN_BUT_BALANCE_MOVED")
             .when(F.col("n_legs") == 0, "no_txn_flat").otherwise("has_txn"))
         .groupBy("bucket").agg(F.count(F.lit(1)).alias("n"),
             F.expr("percentile_approx(abs(bal_delta), 0.5)").alias("p50_abs_delta")).toPandas(),
         "D4_unexplained_movement",
         "*** the real ceiling on transaction-derived features (v3: 6.6%) ***")

---
# PHASE E — Metrics, labels, power

## E1 · `off_pnc_ach_orig_share` — the treasury-services signal

**New, and stronger than the brief's priority metric.** Of the outbound ACH a
customer originates, what share is carried by a bank that is not PNC. In the
v3 window that was $144.8B of $707.2B — 20.5%.

Why this beats same-name outflow: it is a wallet-share **ratio** rather than a
proxy, it needs no name matching, and a relationship manager can read it
without explanation. A customer moving ACH origination away from PNC is losing
PNC treasury services, which is closer to what "leaving" means than a balance
decline is.

NaN where the customer originated no outbound ACH — never zero, which would
read as "all of it with PNC".

In [ ]:
with timed("E1 ACH origination wallet share"):
    ach = cm.filter(F.col("off_pnc_ach_orig_share").isNotNull())

    save(ach.groupBy("month").agg(
            F.count(F.lit(1)).alias("n_customers"),
            F.mean("off_pnc_ach_orig_share").alias("mean_share"),
            F.expr("percentile_approx(off_pnc_ach_orig_share, 0.5)").alias("p50"),
            F.expr("percentile_approx(off_pnc_ach_orig_share, 0.9)").alias("p90"),
            F.mean((F.col("off_pnc_ach_orig_share") > 0).cast("double")).alias("share_any_nonpnc"),
            (F.sum("out_ach_nonpnc") / F.sum("out_ach_total")).alias("dollar_weighted")
         ).orderBy("month").toPandas(),
         "E1_ach_orig_share_by_month", "*** the treasury-services wallet-share series ***")

    # The delta is the signal, not the level — same discipline as same-name.
    w1 = W.partitionBy("cust_pwr_id").orderBy("month")
    ach_d = (ach.withColumn("prev", F.lag("off_pnc_ach_orig_share", 1).over(w1))
                .withColumn("prev3", F.lag("off_pnc_ach_orig_share", 3).over(w1))
                .withColumn("d1", F.col("off_pnc_ach_orig_share") - F.col("prev"))
                .withColumn("d3", F.col("off_pnc_ach_orig_share") - F.col("prev3")))
    ach_d.write.mode("overwrite").parquet(path("ach_orig_share"))

    save(ach_d.filter(F.col("d3").isNotNull()).agg(
            F.count(F.lit(1)).alias("n"),
            F.expr("percentile_approx(d3, 0.5)").alias("p50_d3"),
            F.expr("percentile_approx(d3, 0.9)").alias("p90_d3"),
            F.expr("percentile_approx(d3, 0.99)").alias("p99_d3"),
            F.mean((F.col("d3") > 0.10).cast("double")).alias("share_shift_gt_10pt"),
            F.mean((F.col("d3") > 0.25).cast("double")).alias("share_shift_gt_25pt")).toPandas(),
         "E1_ach_orig_delta",
         "share_shift_gt_25pt is the candidate alert rate — compare it against the 30% rule's 39%")

## E2 · Same-name outflow — level, delta, and the false positive

v3 measured a base rate of **7.8% of customers per month** (11% of dollars),
flat. As a level the flag fires on ~3,100 customers permanently, because most
corporates simply hold accounts elsewhere. The signal is a **new** same-name
destination or a step change in share.

v3 also exposed the false-positive mechanism: `TREASURY IRS EFTPS RECV` and
`US TREASURY SINGLE TAXPAYORS` appear among same-name destinations, because tax
payments carry the payer's own name in the counterparty field. Excluded here.

In [ ]:
with timed("E2 same-name level + delta"):
    sn = (edges.filter("ego_dir='OUT' AND cpty_key_src='off_us'")
        .join(F.broadcast(corp.select("cust_pwr_id", "ego_name_norm").dropDuplicates(["cust_pwr_id"])),
              "cust_pwr_id")
        .filter(present("cpty_name"))
        .withColumn("cpty_norm", norm_name(F.col("cpty_name")))
        .withColumn("is_tax_fp",
            F.col("cpty_norm").rlike(TAX_FP_PATTERN) |
            F.coalesce(F.col("cpty_fi"), F.lit("")).rlike(TAX_FP_PATTERN))
        .withColumn("same_name", (F.col("cpty_norm") == F.col("ego_name_norm")) & (~F.col("is_tax_fp"))))

    save(sn.groupBy("month").agg(
            ndv(F.when(F.col("same_name"), F.col("cust_pwr_id"))).alias("n_same_name"),
            ndv("cust_pwr_id").alias("n_total"),
            (F.sum(F.when(F.col("same_name"), F.col("amount")).otherwise(0.0))
             / F.sum("amount")).alias("dollar_share"))
         .withColumn("base_rate", F.col("n_same_name") / F.col("n_total"))
         .orderBy("month").toPandas(),
         "E2_same_name_base_rate", "*** high and flat => the LEVEL is not a signal ***")

    # The delta: a same-name destination appearing for the first time. First-seen
    # month per (customer, counterparty) turns a standing relationship into an event.
    firsts = sn.filter("same_name").groupBy("cust_pwr_id", "cpty_key") \
               .agg(F.min("month").alias("first_month"))
    new_dest = (firsts.filter(F.col("first_month") > F.lit(MONTHS[0]))
                      .groupBy("first_month").agg(ndv("cust_pwr_id").alias("n_customers_new_dest"))
                      .withColumnRenamed("first_month", "month").orderBy("month"))
    save(new_dest.toPandas(), "E2_new_same_name_destination",
         "*** THIS is the event. Compare its rate against the 7.8% standing base rate. ***")

## E3 · Counterparty churn — Group B feasibility

Cheap, and it establishes whether relationship dissolution is measurable before
the edge rhythm module is built. Counterparties active last month and absent
this month, weighted by the dollars they carried.

Only meaningful because keys are **account-derived** (96.3% of dollars): a
disappearance is a real event rather than a spelling change.

In [ ]:
with timed("E3 counterparty churn"):
    cur = (edges.filter("ego_dir='OUT' AND cpty_key_src='off_us' AND key_provenance='account_derived'")
                .select("cust_pwr_id", "cpty_key", "month", "amount"))
    # Shift the prior month FORWARD by one and join on identical key names. A
    # self-join with `cur.col == prv.col` is ambiguous under shared lineage;
    # aligning the month first turns it into an ordinary equi-join.
    prv = (cur.withColumn("month", next_month(F.col("month")))
              .withColumnRenamed("amount", "prev_amount"))

    churn = (cur.join(prv, ["cust_pwr_id", "cpty_key", "month"], "full_outer")
        .groupBy("cust_pwr_id", "month")
        .agg(F.sum(F.when(F.col("amount").isNull(), 1).otherwise(0)).alias("n_lost"),
             F.sum(F.when(F.col("prev_amount").isNull(), 1).otherwise(0)).alias("n_new"),
             F.sum(F.when(F.col("amount").isNull(), F.col("prev_amount")).otherwise(0.0)).alias("lost_dollars"),
             F.sum(F.coalesce(F.col("prev_amount"), F.lit(0.0))).alias("prev_dollars"))
        .withColumn("lost_dollar_share",
                    F.col("lost_dollars") / F.greatest(F.col("prev_dollars"), F.lit(1.0)))
        .filter(F.col("month").isin(MONTHS)))
    churn.write.mode("overwrite").partitionBy("month").parquet(path("cpty_churn"))

    save(spark.read.parquet(path("cpty_churn")).groupBy("month").agg(
            F.count(F.lit(1)).alias("n_customers"),
            F.mean("n_lost").alias("mean_lost"),
            F.mean("lost_dollar_share").alias("mean_lost_dollar_share"),
            F.expr("percentile_approx(lost_dollar_share, 0.9)").alias("p90_lost_share")
         ).orderBy("month").toPandas(),
         "E3_counterparty_churn", "the Group B base rate — a stable series is the null model")

## E4 · Account-activity closure proxy

`latest_num_closed_accounts` is current-state, so it cannot date a closure and
cannot be a time-varying label. A monthly version is the pinned ask. Meanwhile
`n_active_accts` gives a dated behavioural closure. v3 measured corr 0.813 and
68% exact agreement with the stated account count.

In [ ]:
with timed("E4 closure proxy"):
    save(cm.groupBy("cust_pwr_id").agg(F.max("n_active_accts").alias("txn_max_accts"))
           .join(dep_w.select(F.col(DEP["cust"]).alias("cust_pwr_id"),
                              F.col(DEP["n_acct"]).cast("int").alias("dep_n_accts")), "cust_pwr_id")
           .agg(F.corr("txn_max_accts", "dep_n_accts").alias("corr"),
                F.mean((F.col("txn_max_accts") == F.col("dep_n_accts")).cast("double")).alias("share_exact"),
                F.mean(F.col("txn_max_accts").cast("double")).alias("mean_txn"),
                F.mean(F.col("dep_n_accts").cast("double")).alias("mean_dep")).toPandas(),
         "E4_account_count_agreement")

## E5 · Episode funnel and the rule's confusion matrix

**The headline output.** v3 measured the 30% rule at 39.0% of the book with
~11% precision against departure. That is the finding driving the reply to the
brief, so it is computed as a first-class artefact here rather than by hand.

`series_stopped` is a proxy and is affected by D2b — the precision figure will
move. The order of magnitude will not: a rule firing on ~39% of the corporate
book cannot be a departure detector under any label, because departure base
rates are nowhere near 39%.

In [ ]:
with timed("E5 episode funnel + confusion"):
    w5 = W.partitionBy("cust_pwr_id").orderBy("month")
    ep = (dep.filter(F.col("balance").isNotNull())
        .withColumn("avg3",  F.avg("balance").over(w5.rowsBetween(-2, 0)))
        .withColumn("avg6p", F.avg("balance").over(w5.rowsBetween(-8, -3)))
        .withColumn("n_hist", F.count("balance").over(w5.rowsBetween(-11, 0)))
        .withColumn("fwd3",  F.avg("balance").over(w5.rowsBetween(1, 3)))
        .withColumn("flag",  F.col("avg3") <= (1 - CONFIG["CURRENT_RULE_DROP"]) * F.col("avg6p")))
    ep.write.mode("overwrite").parquet(path("episodes"))
    ep = spark.read.parquet(path("episodes"))

    full = ((F.col("n_hist") >= 12) & (F.col("avg6p") >= CONFIG["BALANCE_FLOOR"])
            & (F.col("fwd3") <= 1.1 * F.col("avg3")))
    rows = []
    for label, cond in [("1_raw_30pct_rule", F.lit(True)),
                        ("2_plus_12mo_history", F.col("n_hist") >= 12),
                        ("3_plus_balance_floor", (F.col("n_hist") >= 12) & (F.col("avg6p") >= CONFIG["BALANCE_FLOOR"])),
                        ("4_plus_persistence", full)]:
        rows.append({"step": label,
                     "n_customers": ep.filter(F.col("flag") & cond).select("cust_pwr_id").distinct().count()})
    funnel = save(pd.DataFrame(rows), "E5_episode_funnel", "the last row is the real sample size")

    survivors = ep.filter(F.col("flag") & full).select("cust_pwr_id").distinct() \
                  .withColumn("rule_fires", F.lit(1))
    truth = pat.select("cust_pwr_id",
                       (F.col("shape") == "STOPPED_BEFORE_END").cast("int").alias("series_stopped"))

    cmx = (dep_ids.join(survivors, "cust_pwr_id", "left").join(truth, "cust_pwr_id", "left")
                  .fillna({"rule_fires": 0, "series_stopped": 0}))
    tab = cmx.groupBy("rule_fires", "series_stopped").count().toPandas()
    save(tab, "E5_rule_confusion", "*** THE HEADLINE ***")

    g = {(int(r.rule_fires), int(r.series_stopped)): int(r["count"]) for _, r in tab.iterrows()}
    tp, fp = g.get((1, 1), 0), g.get((1, 0), 0)
    fn, tn = g.get((0, 1), 0), g.get((0, 0), 0)
    scores = pd.DataFrame([{
        "n_book": tp + fp + fn + tn,
        "n_flagged": tp + fp,
        "flag_rate": (tp + fp) / max(tp + fp + fn + tn, 1),
        "n_stops": tp + fn,
        "stop_rate": (tp + fn) / max(tp + fp + fn + tn, 1),
        "precision": tp / max(tp + fp, 1),
        "recall": tp / max(tp + fn, 1),
        "lift_vs_random": (tp / max(tp + fp, 1)) / max((tp + fn) / max(tp + fp + fn + tn, 1), 1e-9),
    }])
    save(scores, "E5_rule_scores",
         "*** precision ~0.11 in v3. lift_vs_random is the fairest single number to brief. ***")
    R.update({"rule_flag_rate": float(scores["flag_rate"].iloc[0]),
              "rule_precision": float(scores["precision"].iloc[0]),
              "rule_recall": float(scores["recall"].iloc[0]),
              "rule_lift": float(scores["lift_vs_random"].iloc[0])})

    gv = (survivors.join(seen_o, "cust_pwr_id", "left")
        .agg(F.count(F.lit(1)).alias("n_episodes"),
             F.sum(F.coalesce("_out", F.lit(0))).alias("n_outbound_visible")).toPandas())
    gv["coverage"] = gv["n_outbound_visible"] / gv["n_episodes"]
    save(gv, "E5_episodes_graph_visible")
    R["episodes_outbound_visible"] = int(gv["n_outbound_visible"].iloc[0])

## E6 · Candidate labels — with the silence window fixed

v3 returned `rate_txn_silence = 1.000`. The cause: the grid spanned 30 deposit
months against 3 transaction months, so 27 months were trivially silent. The
label was measuring the extraction window, not the customer.

**Fix:** silence is evaluated only on months where a full forward window of
transaction coverage exists. Outside that range the label is NULL, not 0 — an
unobserved month is not a quiet one.

In [ ]:
with timed("E6 labels"):
    txn_months = sorted([r["month"] for r in cm.select("month").distinct().collect()])
    k = CONFIG["SILENCE_MONTHS"]
    valid_months = txn_months[:len(txn_months) - (k - 1)] if len(txn_months) >= k else []
    print(f"txn coverage: {len(txn_months)} months · silence evaluable on {len(valid_months)}")
    if not valid_months:
        raise ValueError("Transaction window is shorter than SILENCE_MONTHS — widen MONTH_MIN/MAX.")

    grid = (dep.filter(F.col("month").isin(txn_months))
               .select("cust_pwr_id", "month")
               .join(cm.select("cust_pwr_id", "month", "n_legs"), ["cust_pwr_id", "month"], "left")
               .fillna({"n_legs": 0}))
    w6 = W.partitionBy("cust_pwr_id").orderBy("month")
    silence = (grid.withColumn("fwd", F.sum("n_legs").over(w6.rowsBetween(0, k - 1)))
        .filter(F.col("month").isin(valid_months))     # <-- the fix
        .withColumn("silent", (F.col("fwd") == 0).cast("int"))
        .groupBy("cust_pwr_id").agg(F.max("silent").alias("txn_silence")))

    labels = (dep_ids
        .join(silence, "cust_pwr_id", "left")          # NULL where unobservable
        .join(truth, "cust_pwr_id", "left")
        .join(survivors.withColumnRenamed("rule_fires", "balance_30pct"), "cust_pwr_id", "left")
        .fillna({"series_stopped": 0, "balance_30pct": 0}))
    labels.write.mode("overwrite").parquet(path("labels"))

    save(labels.groupBy("balance_30pct", "series_stopped", "txn_silence").count()
               .orderBy(F.desc("count")).limit(20).toPandas(),
         "E6_label_agreement", "*** how much the three targets disagree ***")
    save(labels.agg(
            F.mean(F.col("balance_30pct").cast("double")).alias("rate_balance_30pct"),
            F.mean(F.col("series_stopped").cast("double")).alias("rate_series_stopped"),
            F.mean(F.col("txn_silence").cast("double")).alias("rate_txn_silence"),
            F.mean(F.col("txn_silence").isNull().cast("double")).alias("share_silence_unobservable")).toPandas(),
         "E6_label_prevalence",
         "*** rate_txn_silence near 1.0 means the window fix did not take ***")
    R["rate_txn_silence"] = float(
        labels.agg(F.mean(F.col("txn_silence").cast("double"))).first()[0] or 0)

## E7 · Summary

In [ ]:
save(pd.DataFrame(list(R.items()), columns=["metric", "value"]), "E7_SUMMARY")
save(pd.DataFrame(TIMINGS), "E7_TIMINGS")

print("""
READING THE SUMMARY
-------------------
rule_flag_rate / rule_precision / rule_lift
    v3: 0.390 / 0.113 / 1.57x. The lift is the fairest single number to brief —
    it says the rule is barely better than flagging at random. Everything else in
    the programme is an attempt to beat this, so it is the benchmark, not a result.

stop_share_last_8mo   >0.9 => the deposit panel is a current-book snapshot and
                      departures before the cutover are missing. ESCALATE. All
                      historical episode work is survivor-only until resolved.

rate_txn_silence      If still ~1.0 after the window fix, the transaction window
                      does not overlap the deposit panel usefully. Check
                      A4_monthly_volume against D2b_month_coverage.

cpty_share_5plus      v3: 0.0020. Group C is undefined for 99.8% of counterparties.

E1_ach_orig_delta     share_shift_gt_25pt is the candidate alert rate. If it is
                      2-5% of the book against the 30% rule's 39%, that alone is
                      the argument for replacing the rule.

D4 c3/c4              The log-space correlations are the honest test of whether
                      deposits and payments are one ledger. Do NOT brief c1.

NEXT
    1. Read D2b first. It gates everything historical.
    2. E5_rule_scores goes to TM Ops as-is.
    3. E1 + E2_new_same_name_destination are the two candidate replacements.
       Score both against series_stopped the same way E5 scores the 30% rule.
    4. Only then build features.
""")